In [ ]:
# this notebook calls functions in dimensionality_reduction module

In [ ]:
import os
import sys
from glob import glob

module_path = os.path.abspath(os.path.join('../module/'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from prostate_recurrence.data_exploration import dimensionality_reduction

In [ ]:
from glob import glob
import pandas as pd

## read data

### ... define paths to data that contain tile-level quantitative features

In [ ]:
# subfolder_analysis = "20210831_outputs_test_analysis"; slides = ['C032080_B05', 'C023006_B01'] # both have around 100 slides; in pilot analysis 2021.08

In [ ]:
# subfolder_analysis = "20210904_outputs_test_analysis"; slides = ['C022134_B02', 'C042032_B01', 'C042101_B02', 'C042057_B03', 'C023002_B01', 'C043020_B01']

In [ ]:
subfolder_analysis = "20210907_outputs_test_analysis"; slides = ['C042057_B03']

In [ ]:
path_part_1 = '/Volumes/proj-sahai-tme-ml/working/processed_data/feature_engineering/clinical/prostate/chiip_cohort/slide/'

In [ ]:
path_part_2 = 'tile_size_1024/whole_slide/PSR/deconvolutions/psr/inverted_grayscale/tile_level_features/'

In [ ]:
data_file_pattern = 'features_out.csv'

### ... read 

In [ ]:
all_paths_to_data = []
for slide in slides:
    paths_to_data = glob(
        os.path.join(
            path_part_1,
            slide,
            path_part_2,
            "*",
            data_file_pattern
        )
    )
    all_paths_to_data.extend(
        paths_to_data
    )

In [ ]:
all_data_labeled = dimensionality_reduction.read_data(
    all_paths_to_data
)

### ... save raw data as .csv

In [ ]:
output_path_save_raw = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}/",
    "all_data_labeled.csv"
)

In [ ]:
all_data_labeled.to_csv(
    output_path_save_raw,
    index=False
)

## process data

### ... read raw data from .csv

In [ ]:
input_path_save_raw = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}/",
    "all_data_labeled.csv"
)
all_data_labeled = pd.read_csv(
    input_path_save_raw
)

### ... keep features from data

In [ ]:
features = all_data_labeled[
    [
        col
        for col in all_data_labeled.columns
        if col not in ["slide", "image_tile", "label"]
    ]
]
features.shape

### ... processing

In [ ]:
(
    processed_features,
    standardisation_attributes
) = dimensionality_reduction.process_data(features)

In [ ]:
processed_data_labeled = pd.concat(
    [
        processed_features, 
        all_data_labeled[['slide', 'image_tile', 'label']]
    ],
    axis=1
)

### ... save processed data as .csv

In [ ]:
output_path_save_processed = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "processed_data_labeled.csv"
)

In [ ]:
processed_data_labeled.to_csv(
    output_path_save_processed,
    index=False
)

### ... save transformation attributes

In [ ]:
output_path_save_transformation_attributes = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "processed_data_standardisation_attributes.csv"
)

In [ ]:
standardisation_attributes.to_csv(
    output_path_save_transformation_attributes,
    index=False
)

## PCA

### ... read processed data from .csv

In [ ]:
input_path_save_processed = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "processed_data_labeled.csv"
)
processed_data_labeled = pd.read_csv(
    input_path_save_processed
)

### performed on all data points

In [ ]:
output_directory_save_pca = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "pca_results"
)
os.makedirs(output_directory_save_pca, exist_ok=True)

In [ ]:
pca_attributes = dimensionality_reduction.perform_pca(
    output_directory=output_directory_save_pca,
    data=processed_data_labeled
)

In [ ]:
pca_attributes.keys()

### performed on defined tumour or normal

In [ ]:
tumour_labels = pd.read_csv(
    "/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m08m09__pilot_analysis/C042057_B03/tumour_labels.csv"
)

In [ ]:
tumour_labels.head()

In [ ]:
image_tiles = []
for path in tumour_labels.path_to_image_tile:
    image_tile = os.path.basename(path).split('.')[0]
    image_tiles.append(image_tile)
tumour_labels['image_tile'] = image_tiles

In [ ]:
subset_processed_data_labeled = processed_data_labeled.loc[
    processed_data_labeled.image_tile.isin(
        tumour_labels.loc[
            tumour_labels.label.isin(
                ['Tumour', 'Normal']
            )
        ].image_tile.values
    )
]

In [ ]:
subset_processed_data_labeled.shape

### ... keep all features

In [ ]:
output_directory_save_pca = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "pca_results_subset_with_tumour_annot"
)
os.makedirs(output_directory_save_pca, exist_ok=True)

In [ ]:
pca_attributes = dimensionality_reduction.perform_pca(
    output_directory=output_directory_save_pca,
    data=subset_processed_data_labeled
)

### ... keep only certain features

In [ ]:
keywords_exclusion = [
    'conv', 'HDM', 'derivative',
    'jensen shannon', 'Hellinger',
    'Bhattacharyya'
]

features_exclusion = []
for col in subset_processed_data_labeled.columns:
    flag_exclusion = False
    for keyword in keywords_exclusion:
        if keyword in col:
            flag_exclusion = True
    if flag_exclusion:
        features_exclusion.append(col)
        
print(features_exclusion)
print(len(features_exclusion))

In [ ]:
subset_processed_data_labeled_exclude_certain_features = subset_processed_data_labeled[
    [
        col for col in subset_processed_data_labeled.columns 
        if col not in features_exclusion
    ]
]
subset_processed_data_labeled_exclude_certain_features.head()

In [ ]:
output_path_save_processed_with_exclusion = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "subset_processed_data_labeled_exclude_certain_features.csv"
)
subset_processed_data_labeled_exclude_certain_features.to_csv(
    output_path_save_processed_with_exclusion,
    index=False
)

In [ ]:
output_directory_save_pca = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "pca_results_subset_with_tumour_annot_exclude_certain_features"
)
os.makedirs(output_directory_save_pca, exist_ok=True)


In [ ]:
pca_attributes = dimensionality_reduction.perform_pca(
    output_directory=output_directory_save_pca,
    data=subset_processed_data_labeled_exclude_certain_features
)

## t-SNE

### ... read processed data from .csv

In [ ]:
input_path_save_processed = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "processed_data_labeled.csv"
)
processed_data_labeled = pd.read_csv(
    input_path_save_processed
)

In [ ]:
output_directory_save_tsne = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "tsne_results"
)
os.makedirs(output_directory_save_tsne, exist_ok=True)

In [ ]:
tsne_attributes = dimensionality_reduction.perform_tsne(
    output_directory=output_directory_save_tsne,
    data=processed_data_labeled
)

## UMAP

### ... read processed data from .csv

In [ ]:
input_path_save_processed = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "processed_data_labeled.csv"
)
processed_data_labeled = pd.read_csv(
    input_path_save_processed
)

In [ ]:
output_directory_save_umap = os.path.join(
    f"/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/{subfolder_analysis}",
    "umap_results"
)
os.makedirs(output_directory_save_umap, exist_ok=True)

In [ ]:
umap_attributes = dimensionality_reduction.perform_umap(
    output_directory=output_directory_save_umap,
    data=processed_data_labeled
)